# 11_xgboost_cv_final

This notebook is the cleaned-up final XGBoost pipeline for the project.

Main fixes compared with earlier notebooks:
- Uses **Stratified 5-Fold CV** instead of one reused holdout split
- Uses **OOF ROC-AUC** as the main internal score
- Uses **native categorical handling** in XGBoost
- Keeps original features and **adds** engineered features
- Removes the misleading `scale_pos_weight` setup


In [2]:
# 11_xgboost_cv_final_fixed.ipynb
# 修复版本：使用更兼容的方式处理类别特征

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

print("="*60)
print("11_xgboost_cv_final: XGBoost with Cross Validation")
print("="*60)

# 路径设置
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models/xgb_cv_final'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
LOG_DIR = PROJECT_ROOT / 'logs'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'diagnosed_diabetes'
N_SPLITS = 5
RANDOM_STATE = 42

print(f"\n项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"模型保存目录: {MODEL_DIR}")


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """添加工程特征（保留原始特征）"""
    df = df.copy()
    
    # 交互特征（保留原始age和family_history_diabetes）
    df['age_family_history'] = df['age'] * df['family_history_diabetes']
    df['age_bmi'] = df['age'] * df['bmi']
    
    # 心血管风险评分
    df['cardio_risk_score'] = (
        df['family_history_diabetes'] * 3 +
        df['hypertension_history'] * 2 +
        df['cardiovascular_history'] * 2
    )
    
    # 胆固醇指标
    df['cholesterol_ratio'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 1)
    df['non_hdl_cholesterol'] = df['cholesterol_total'] - df['hdl_cholesterol']
    
    return df


def encode_categorical(df: pd.DataFrame) -> pd.DataFrame:
    """手动编码类别特征（兼容性更好）"""
    df = df.copy()
    
    # 需要编码的类别特征
    categorical_cols = ['gender', 'ethnicity', 'education_level', 
                        'income_level', 'smoking_status', 'employment_status']
    
    # 对每个类别特征进行Label Encoding
    for col in categorical_cols:
        if col in df.columns:
            le = LabelEncoder()
            # 处理可能的缺失值
            df[col] = df[col].fillna('Unknown')
            df[col] = le.fit_transform(df[col].astype(str))
    
    return df


def build_model(seed: int = RANDOM_STATE) -> XGBClassifier:
    """构建XGBoost模型"""
    return XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',           # 直方图算法，更快
        n_estimators=3000,            # 最大树数，早停会提前停止
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        random_state=seed,
        n_jobs=-1,
        early_stopping_rounds=100,
        verbosity=0
    )


def main():
    # 1. 加载数据
    print("\n1. 加载数据...")
    train_df = pd.read_csv(DATA_DIR / 'train.csv')
    test_df = pd.read_csv(DATA_DIR / 'test.csv')
    sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
    
    print(f"训练集: {train_df.shape}")
    print(f"测试集: {test_df.shape}")
    
    # 获取列名
    id_col = sample_sub.columns[0]
    pred_col = sample_sub.columns[1]
    
    # 2. 准备数据
    print("\n2. 准备数据...")
    X = train_df.drop(columns=[id_col, TARGET])
    y = train_df[TARGET].astype(int)
    X_test = test_df.drop(columns=[id_col])
    
    print(f"原始特征数: {X.shape[1]}")
    
    # 3. 添加工程特征
    print("\n3. 添加工程特征...")
    X = add_engineered_features(X)
    X_test = add_engineered_features(X_test)
    print(f"添加后特征数: {X.shape[1]}")
    
    # 4. 编码类别特征（兼容版本）
    print("\n4. 编码类别特征...")
    X = encode_categorical(X)
    X_test = encode_categorical(X_test)
    print(f"编码后特征数: {X.shape[1]}")
    
    # 确保测试集和训练集列一致
    missing_cols = set(X.columns) - set(X_test.columns)
    for col in missing_cols:
        X_test[col] = 0
    X_test = X_test[X.columns]
    
    print(f"训练集特征数: {X.shape[1]}")
    print(f"测试集特征数: {X_test.shape[1]}")
    
    # 5. 5折交叉验证
    print("\n5. 开始5折交叉验证...")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    
    oof_pred = np.zeros(len(X), dtype=float)
    test_pred = np.zeros(len(X_test), dtype=float)
    fold_scores = []
    best_iterations = []
    feature_importance_sum = np.zeros(X.shape[1])
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        print(f"\n{'='*40}")
        print(f"Fold {fold}/{N_SPLITS}")
        print(f"{'='*40}")
        
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
        
        print(f"训练集: {len(X_train)} 样本")
        print(f"验证集: {len(X_valid)} 样本")
        
        # 训练模型
        model = build_model(seed=RANDOM_STATE + fold)
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )
        
        # 验证集预测
        valid_pred = model.predict_proba(X_valid)[:, 1]
        fold_auc = roc_auc_score(y_valid, valid_pred)
        oof_pred[valid_idx] = valid_pred
        fold_scores.append(fold_auc)
        
        # 获取最佳迭代次数
        best_iter = model.best_iteration if hasattr(model, 'best_iteration') else None
        best_iterations.append(best_iter)
        
        print(f"验证集 AUC: {fold_auc:.6f}")
        if best_iter:
            print(f"最佳迭代次数: {best_iter}")
        
        # 累积特征重要性
        feature_importance_sum += model.feature_importances_
        
        # 测试集预测（累加）
        test_pred += model.predict_proba(X_test)[:, 1] / N_SPLITS
        
        # 保存模型
        model.save_model(MODEL_DIR / f'xgb_fold_{fold}.json')
        print(f"模型已保存: {MODEL_DIR / f'xgb_fold_{fold}.json'}")
    
    # 6. 计算最终结果
    print("\n" + "="*60)
    print("交叉验证结果汇总")
    print("="*60)
    
    oof_auc = roc_auc_score(y, oof_pred)
    mean_fold_auc = np.mean(fold_scores)
    std_fold_auc = np.std(fold_scores)
    
    print(f"\n各折 AUC: {[round(s, 6) for s in fold_scores]}")
    print(f"平均 AUC: {mean_fold_auc:.6f}")
    print(f"标准差: {std_fold_auc:.6f}")
    print(f"OOF AUC: {oof_auc:.6f}")
    
    # 7. 特征重要性
    print("\n7. 特征重要性分析...")
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': feature_importance_sum / N_SPLITS
    }).sort_values('importance', ascending=False)
    
    print("\nTop 10 最重要特征:")
    print(feature_importance.head(10).to_string(index=False))
    
    # 8. 保存结果
    print("\n8. 保存结果...")
    
    # 保存OOF预测
    oof_df = pd.DataFrame({
        id_col: train_df[id_col],
        'y_true': y,
        'oof_pred': oof_pred
    })
    oof_df.to_csv(LOG_DIR / '11_xgboost_cv_oof_predictions.csv', index=False)
    print(f"✓ OOF预测已保存: {LOG_DIR / '11_xgboost_cv_oof_predictions.csv'}")
    
    # 保存特征重要性
    feature_importance.to_csv(LOG_DIR / '11_xgboost_cv_feature_importance.csv', index=False)
    print(f"✓ 特征重要性已保存: {LOG_DIR / '11_xgboost_cv_feature_importance.csv'}")
    
    # 保存提交文件
    submission = sample_sub.copy()
    submission[pred_col] = test_pred
    submission.to_csv(SUBMISSION_DIR / '11_xgboost_cv_submission.csv', index=False)
    print(f"✓ 提交文件已保存: {SUBMISSION_DIR / '11_xgboost_cv_submission.csv'}")
    
    # 9. 保存实验总结
    print("\n9. 保存实验总结...")
    
    summary = {
        'experiment': '11_xgboost_cv_final',
        'n_splits': N_SPLITS,
        'random_state': RANDOM_STATE,
        'fold_aucs': fold_scores,
        'mean_fold_auc': float(mean_fold_auc),
        'std_fold_auc': float(std_fold_auc),
        'oof_auc': float(oof_auc),
        'best_iterations': best_iterations,
        'feature_count': X.shape[1],
        'params': {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'tree_method': 'hist',
            'n_estimators': 3000,
            'learning_rate': 0.05,
            'max_depth': 5,
            'min_child_weight': 5,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'reg_lambda': 2.0,
            'early_stopping_rounds': 100
        },
        'improvements': [
            '使用5折交叉验证替代固定holdout分割',
            '保留原始特征，同时添加交互特征',
            '使用LabelEncoder处理类别特征（兼容性更好）',
            '用OOF AUC作为模型评估指标',
            '早停机制防止过拟合'
        ]
    }
    
    with open(LOG_DIR / '11_xgboost_cv_summary.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    print(f"✓ 实验总结已保存: {LOG_DIR / '11_xgboost_cv_summary.json'}")
    
    # 10. 打印最终总结
    print("\n" + "="*60)
    print("实验完成！")
    print("="*60)
    print(f"""
最终结果:
- 5折交叉验证平均 AUC: {mean_fold_auc:.6f} (±{std_fold_auc:.6f})
- OOF AUC: {oof_auc:.6f}
- 提交文件: {SUBMISSION_DIR / '11_xgboost_cv_submission.csv'}

改进点:
1. ✓ 使用5折交叉验证，避免对固定验证集过拟合
2. ✓ 保留原始特征，同时添加工程特征
3. ✓ 使用LabelEncoder处理类别特征（兼容性更好）
4. ✓ 用OOF AUC作为模型评估指标
5. ✓ 早停机制自动控制树数量
    """)
    
    # 显示特征重要性前5
    print("\nTop 5 最重要特征:")
    for i, row in feature_importance.head(5).iterrows():
        print(f"  {i+1}. {row['feature']}: {row['importance']:.4f}")


if __name__ == "__main__":
    main()

11_xgboost_cv_final: XGBoost with Cross Validation

项目根目录: C:\Users\LENOVO\Desktop\DSAI4203_MachineLearning_Project_revised\DSAI4203_MachineLearning_Project-main
数据目录: C:\Users\LENOVO\Desktop\DSAI4203_MachineLearning_Project_revised\DSAI4203_MachineLearning_Project-main\data
模型保存目录: C:\Users\LENOVO\Desktop\DSAI4203_MachineLearning_Project_revised\DSAI4203_MachineLearning_Project-main\models\xgb_cv_final

1. 加载数据...
训练集: (700000, 26)
测试集: (300000, 25)

2. 准备数据...
原始特征数: 24

3. 添加工程特征...
添加后特征数: 29

4. 编码类别特征...
编码后特征数: 29
训练集特征数: 29
测试集特征数: 29

5. 开始5折交叉验证...

Fold 1/5
训练集: 560000 样本
验证集: 140000 样本
验证集 AUC: 0.726880
最佳迭代次数: 1587
模型已保存: C:\Users\LENOVO\Desktop\DSAI4203_MachineLearning_Project_revised\DSAI4203_MachineLearning_Project-main\models\xgb_cv_final\xgb_fold_1.json

Fold 2/5
训练集: 560000 样本
验证集: 140000 样本
验证集 AUC: 0.725317
最佳迭代次数: 1577
模型已保存: C:\Users\LENOVO\Desktop\DSAI4203_MachineLearning_Project_revised\DSAI4203_MachineLearning_Project-main\models\xgb_cv_final\xgb_fold_2.json

